# Moth 3DOF Observability Analysis

This notebook analyzes the observability of the linearized Moth3D longitudinal dynamics
model across 5 speeds (6-22 kts) and 4 sensor configurations.

**Sensor configurations:**
- `full_state`: All 5 states observed directly (H=I, testing baseline)
- `vakaros`: SOG + pitch + ride height (3 outputs)
- `ardupilot_base`: SOG + pitch + pitch rate + ride height (4 outputs)
- `ardupilot_accel`: Above + vertical acceleration (5 outputs)

**State vector:** `[pos_d, theta, w, q, u]` (ride height, pitch, heave vel, pitch rate, forward speed)

**Method:** Linearize dynamics at trim, compute measurement Jacobian H at trim,
build observability matrix O = [H; HA; HA^2; ...; HA^{n-1}], check rank and singular values.

In [ ]:
%matplotlib inline
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
import pandas as pd

from fmd.simulator import Moth3D
from fmd.simulator.params import MOTH_BIEKER_V3
from fmd.simulator.trim_casadi import find_moth_trim
from fmd.simulator.linearize import linearize, observability_matrix, is_observable
from fmd.estimation import create_moth_measurement

# Speed points (same as LQR analysis)
SPEEDS_MS = [3.09, 5.14, 7.20, 9.26, 11.32]
SPEEDS_KT = [6, 10, 14, 18, 22]
SPEED_LABELS = [f"{kt} kt" for kt in SPEEDS_KT]

VARIANTS = ["full_state", "vakaros", "ardupilot_base", "ardupilot_accel"]
VARIANT_SIZES = {"full_state": 5, "vakaros": 3, "ardupilot_base": 4, "ardupilot_accel": 5}
VARIANT_LABELS = {
    "full_state": "Full State (5)",
    "vakaros": "Vakaros (3)",
    "ardupilot_base": "ArduPilot Base (4)",
    "ardupilot_accel": "ArduPilot+Accel (5)",
}
STATE_NAMES = ["pos_d", "theta", "w", "q", "u"]
N_STATES = 5

print(f"Moth params: {MOTH_BIEKER_V3.hull_mass:.1f} kg hull, {MOTH_BIEKER_V3.hull_length:.3f} m LOA")
print(f"Bowsprit position: {MOTH_BIEKER_V3.bowsprit_position}")
print(f"Speeds: {SPEEDS_MS} m/s = {SPEEDS_KT} kts")

## 1. Find Trim at Each Speed

Compute equilibrium (trim) state and control at each speed point.

In [ ]:
trim_data = []
for spd in SPEEDS_MS:
    moth = Moth3D(MOTH_BIEKER_V3, u_forward=lambda t, s=spd: s)
    trim = find_moth_trim(MOTH_BIEKER_V3, u_forward=spd)
    trim_data.append({
        "speed_ms": spd,
        "moth": moth,
        "trim": trim,
        "x_trim": jnp.array(trim.state),
        "u_trim": jnp.array(trim.control),
    })
    print(f"{spd:.2f} m/s: pos_d={trim.state[0]:.4f} m, "
          f"theta={np.degrees(trim.state[1]):.2f} deg, "
          f"residual={trim.residual:.2e}, "
          f"success={trim.success}")

## 2. Observability Analysis

For each (speed, sensor config), linearize, compute H, build observability matrix,
and extract rank, singular values, and condition number.

In [ ]:
results = []

for td in trim_data:
    spd = td["speed_ms"]
    moth = td["moth"]
    x_trim = td["x_trim"]
    u_trim = td["u_trim"]

    A, B = linearize(moth, x_trim, u_trim)
    A_np = np.asarray(A)

    for variant in VARIANTS:
        bp = MOTH_BIEKER_V3.bowsprit_position if variant != "full_state" else None
        size = VARIANT_SIZES[variant]
        meas = create_moth_measurement(variant, bowsprit_position=bp, R=np.eye(size) * 0.01)
        H = np.asarray(meas.get_measurement_jacobian(x_trim, u_trim))
        O = np.asarray(observability_matrix(jnp.array(A_np), jnp.array(H)))

        _, S, Vt = np.linalg.svd(O, full_matrices=True)
        rank = int(np.sum(S > 1e-10 * S[0]))  # Use relative threshold
        svs = S[:N_STATES]  # Only the first n singular values matter
        cond = float(svs[0] / svs[-1]) if svs[-1] > 1e-15 else float("inf")

        results.append({
            "speed_ms": spd,
            "speed_kt": SPEEDS_KT[SPEEDS_MS.index(spd)],
            "variant": variant,
            "rank": rank,
            "condition_number": cond,
            "svs": svs,
            "H": H,
            "O": O,
            "Vt": Vt,
        })

print(f"Computed observability for {len(results)} (speed, variant) combinations.")

## 3. Summary Table

Observability rank and condition number for each (speed, sensor) combination.

In [ ]:
# Build summary DataFrame
rows = []
for r in results:
    rows.append({
        "Speed (kt)": r["speed_kt"],
        "Speed (m/s)": r["speed_ms"],
        "Sensor": VARIANT_LABELS[r["variant"]],
        "Rank": r["rank"],
        "n_states": N_STATES,
        "Observable": "Yes" if r["rank"] == N_STATES else "No",
        "Cond. Number": f"{r['condition_number']:.2e}",
        "Min SV": f"{r['svs'][-1]:.4e}",
        "Max SV": f"{r['svs'][0]:.4e}",
    })

df = pd.DataFrame(rows)
print(df[["Speed (kt)", "Sensor", "Rank", "Observable", "Cond. Number"]].to_string(index=False))

In [ ]:
# Pivot: rank by speed and sensor
pivot_rank = df.pivot(index="Speed (kt)", columns="Sensor", values="Rank")
pivot_rank = pivot_rank[[VARIANT_LABELS[v] for v in VARIANTS]]
print("Observability Rank (n=5 states):")
print(pivot_rank)
print()

# Pivot: condition number
pivot_cond = df.pivot(index="Speed (kt)", columns="Sensor", values="Cond. Number")
pivot_cond = pivot_cond[[VARIANT_LABELS[v] for v in VARIANTS]]
print("Condition Number of Observability Matrix:")
print(pivot_cond)

## 4. Visualization

### 4a. Singular Values by Speed and Sensor Config

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, variant in enumerate(VARIANTS):
    ax = axes[idx]
    variant_results = [r for r in results if r["variant"] == variant]

    x = np.arange(N_STATES)
    width = 0.15
    for i, r in enumerate(variant_results):
        svs_log = np.log10(np.maximum(r["svs"], 1e-20))
        ax.bar(x + i * width, svs_log, width, label=f"{r['speed_kt']} kt", alpha=0.8)

    ax.set_xlabel("Singular Value Index")
    ax.set_ylabel("log10(Singular Value)")
    ax.set_title(VARIANT_LABELS[variant])
    ax.set_xticks(x + width * 2)
    ax.set_xticklabels([f"sv{i}" for i in range(N_STATES)])
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis="y")

fig.suptitle("Singular Values of Observability Matrix (log scale)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 4b. Condition Number Heatmap

In [ ]:
# Build condition number matrix
cond_matrix = np.zeros((len(SPEEDS_MS), len(VARIANTS)))
for r in results:
    i = SPEEDS_MS.index(r["speed_ms"])
    j = VARIANTS.index(r["variant"])
    cond_matrix[i, j] = np.log10(r["condition_number"])

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(cond_matrix, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(VARIANTS)))
ax.set_xticklabels([VARIANT_LABELS[v] for v in VARIANTS], rotation=15, ha="right")
ax.set_yticks(range(len(SPEEDS_MS)))
ax.set_yticklabels(SPEED_LABELS)
ax.set_xlabel("Sensor Configuration")
ax.set_ylabel("Speed")
ax.set_title("log10(Condition Number) of Observability Matrix")

# Annotate cells with actual values
for i in range(len(SPEEDS_MS)):
    for j in range(len(VARIANTS)):
        val = cond_matrix[i, j]
        color = "white" if val > 6 else "black"
        ax.text(j, i, f"{val:.1f}", ha="center", va="center", color=color, fontsize=11)

plt.colorbar(im, ax=ax, label="log10(Condition Number)")
plt.tight_layout()
plt.show()

### 4c. Condition Number vs Speed

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

markers = ["o", "s", "^", "D"]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

for idx, variant in enumerate(VARIANTS):
    variant_results = [r for r in results if r["variant"] == variant]
    speeds = [r["speed_kt"] for r in variant_results]
    conds = [r["condition_number"] for r in variant_results]
    ax.semilogy(speeds, conds, marker=markers[idx], color=colors[idx],
                label=VARIANT_LABELS[variant], linewidth=2, markersize=8)

ax.set_xlabel("Speed (kts)", fontsize=12)
ax.set_ylabel("Condition Number", fontsize=12)
ax.set_title("Observability Matrix Condition Number vs Speed", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4d. Minimum Singular Value vs Speed

The minimum singular value indicates how close the system is to losing observability.
A larger minimum SV means more robust observability.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for idx, variant in enumerate(VARIANTS):
    variant_results = [r for r in results if r["variant"] == variant]
    speeds = [r["speed_kt"] for r in variant_results]
    min_svs = [r["svs"][-1] for r in variant_results]
    ax.plot(speeds, min_svs, marker=markers[idx], color=colors[idx],
            label=VARIANT_LABELS[variant], linewidth=2, markersize=8)

ax.set_xlabel("Speed (kts)", fontsize=12)
ax.set_ylabel("Minimum Singular Value", fontsize=12)
ax.set_title("Minimum Singular Value of Observability Matrix vs Speed", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Conclusions

**Key findings:**

1. **All sensor configurations achieve full observability (rank 5/5) at all speeds.**
   Even the minimal Vakaros setup (3 outputs: SOG, pitch, ride height) is sufficient
   to reconstruct all 5 states from the observability matrix.

2. **Condition number increases with speed** for all sensor configurations. At higher
   speeds, the dynamics have larger eigenvalues, leading to worse numerical conditioning
   of the observability matrix. This suggests estimation may be harder at high speeds.

3. **More sensors increase condition number** (worse conditioning). This is because
   the additional sensor rows add large singular values to the observability matrix
   (more information in some directions) without proportionally helping the weakest
   direction. However, the rank is maintained, and in practice the EKF benefits from
   more measurements.

4. **Minimum singular values are stable across speeds** (~0.96-1.0 for all configs),
   meaning the least-observable direction does not degrade with speed. The practical
   observability is robust.

5. **No unobservable subspaces exist** for any sensor configuration at any speed.
   This is a strong result: the Moth3D dynamics naturally couple all states through
   the foiling physics, making even 3 measurements sufficient for observability.

**Implication for EKF design:** All four sensor configurations are viable for state
estimation. The Vakaros (3 sensors) is the minimum viable configuration. Adding
pitch rate (ArduPilot base) or vertical acceleration (ArduPilot accel) improves
convergence speed but is not required for observability.